# DALE vs STREME vs MEME: Live Head-to-Head Benchmark

**All three tools run live on this machine. No pre-computed results.**

DALE is a 928 KB statically-linked binary (zero dependencies). STREME and MEME come from the MEME Suite.
Each tool receives the same 12 ENCODE K562 ChIP-seq TFs, discovers motifs independently, and all results
are scored with the same AUROC metric on dinucleotide-shuffled negatives.

**Click Runtime → Run All. Total time: ~2 minutes.**

Paper: [doi.org/10.5281/zenodo.21907349](https://doi.org/10.5281/zenodo.21907349)


## 1. Install MEME Suite & Download DALE


In [ ]:
import os, subprocess, sys, time, shutil
import subprocess
import shutil

# --- Install MEME Suite via conda ---
if not os.path.exists('/opt/meme/bin/streme'):
    print('Installing miniconda...')
    subprocess.run('wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh '
                   '-O /tmp/miniconda.sh', shell=True, check=True)
    subprocess.run('bash /tmp/miniconda.sh -b -p $HOME/miniconda 2>&1 | tail -1',
                   shell=True, executable='/bin/bash')
    os.environ['PATH'] = f"$HOME/miniconda/bin:{os.environ.get('PATH', '')}"
    print('Installing MEME Suite (STREME + MEME)...')
    r = subprocess.run('conda install -y -c bioconda -c conda-forge meme-suite 2>&1 | tail -5',
                        shell=True, executable='/bin/bash', capture_output=True, text=True)
    print(r.stdout)
    # Create /opt/meme symlink so DALE benchmark can find them
    meme_bindir = subprocess.check_output('find $HOME/miniconda -name streme -type f 2>/dev/null | head -1',
                                           shell=True, executable='/bin/bash', text=True).strip()
    if meme_bindir:
        meme_parent = os.path.dirname(meme_bindir)
        os.makedirs('/opt/meme/bin', exist_ok=True)
        for tool in ['streme', 'meme', 'fasta-shuffle', 'fasta-get-markov']:
            src = os.path.join(meme_parent, tool)
            if os.path.exists(src):
                os.symlink(src, f'/opt/meme/bin/{tool}')
        print(f'Symlinked MEME Suite to /opt/meme/bin/')

# --- Download DALE ---
if not os.path.isdir('DALE'):
    subprocess.run('rm -rf DALE && git clone -q https://github.com/Travis42/little-scientist-dale.git DALE',
                   shell=True, check=True)
os.chdir('DALE')
subprocess.run(['chmod', '+x', 'DALE'], check=True)

n_tfs = len([f for f in os.listdir('example') if f.endswith('.fa')])
streme_ok = os.path.exists('/opt/meme/bin/streme')
meme_ok = os.path.exists('/opt/meme/bin/meme')

print(f'\n✓ DALE:   {os.path.getsize("DALE")//1024} KB (static binary, zero deps)')
print(f'{"✓" if streme_ok else "✗"} STREME: {"/opt/meme/bin/streme"}')
print(f'{"✓" if meme_ok else "✗"} MEME:   {"/opt/meme/bin/meme"}')
print(f'✓ Data:   {n_tfs} TFs from ENCODE K562 ChIP-seq in example/')

if not streme_ok:
    raise RuntimeError('STREME not found at /opt/meme/bin/streme. Installation may have failed.')



## 2. Compile Benchmark Binary

The benchmark binary runs all three tools on each TF and scores them with identical AUROC evaluation.
We compile from source (C99 + libm only) with custom STREME/MEME paths.


In [ ]:
import os, subprocess
import subprocess
import shutil

os.chdir('src')

# Fix license header (# comments -> /* */) for gcc compatibility
with open('dale.c', 'r') as f:
    lines = f.readlines()
end = 0
for i, line in enumerate(lines):
    if line.startswith('# ') and i < 20:
        end = i + 1
    elif end > 0 and not line.startswith('#'):
        break
if end > 0:
    new_lines = ['/*\n']
    for line in lines[:end]:
        new_lines.append(' * ' + line.lstrip('# ').rstrip('\n') + '\n')
    new_lines.append(' */\n\n')
    new_lines.extend(lines[end:])
    with open('dale.c', 'w') as f:
        f.writelines(new_lines)

# Compile
subprocess.run('cc -std=c99 -O2 -Wall -Wextra -Wno-unused-parameter -c dale.c -o dale.o', shell=True, check=True)
subprocess.run('cc -std=c99 -O2 -Wall -c benchmark.c -o benchmark.o', shell=True, check=True)
subprocess.run('cc -o benchmark dale.o benchmark.o -lm', shell=True, check=True)
shutil.copy('benchmark', '../benchmark')
os.chdir('..')
print(f'Benchmark binary: {os.path.getsize("benchmark")} bytes')



## 3. Run All Three Tools (Live Head-to-Head)

DALE runs first (~0.3s/TF), then STREME (~2-3s/TF), then MEME (~60-120s/TF).
The benchmark scores all three with the same AUROC metric on the same data.


In [ ]:
import subprocess, time, pandas as pd
import pandas as pd

t0 = time.time()
r = subprocess.run(
    ['./benchmark', '--data', 'example/', '--tf', 'ALL'],
    capture_output=True, text=True, timeout=600
)
elapsed = time.time() - t0
print(r.stdout)
if r.stderr and 'warning' not in r.stderr.lower():
    print('STDERR:', r.stderr[:500])
print(f'\nWall time: {elapsed:.1f}s')



## 4. Results & Visualization


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import wilcoxon

# Parse benchmark output
rows = []
for line in r.stdout.strip().split('\n'):
    parts = line.split('\t')
    if len(parts) >= 5 and parts[0] != 'TF':
        try:
            rows.append({
                'TF': parts[0], 'Width': int(parts[1]),
                'AUROC': float(parts[2]), 'Time_s': float(parts[3]),
                'Source': parts[4]
            })
        except (ValueError, IndexError):
            pass

df = pd.DataFrame(rows)
dale = df[df['Source'] == 'ours'].rename(columns={'AUROC': 'DALE_AUROC', 'Time_s': 'DALE_Time'})
streme = df[df['Source'] == 'streme'].rename(columns={'AUROC': 'STREME_AUROC', 'Time_s': 'STREME_Time'})
meme = df[df['Source'] == 'meme'].rename(columns={'AUROC': 'MEME_AUROC', 'Time_s': 'MEME_Time'})

merged = dale[['TF', 'DALE_AUROC', 'DALE_Time']].merge(
    streme[['TF', 'STREME_AUROC', 'STREME_Time']], on='TF'
).merge(
    meme[['TF', 'MEME_AUROC', 'MEME_Time']], on='TF'
)

print(f'Tools compared on {len(merged)} TFs')
print()

# Stats
MD_COLOR = '#2166AC'
ST_COLOR = '#D6604D'
MM_COLOR = '#4DAF4A'

print(f'DALE:   avg AUROC = {merged["DALE_AUROC"].mean():.4f}  avg time = {merged["DALE_Time"].mean():.2f}s/TF')
print(f'STREME: avg AUROC = {merged["STREME_AUROC"].mean():.4f}  avg time = {merged["STREME_Time"].mean():.2f}s/TF')
print(f'MEME:   avg AUROC = {merged["MEME_AUROC"].mean():.4f}  avg time = {merged["MEME_Time"].mean():.2f}s/TF')
print()

d_vs_s = merged['DALE_AUROC'].values - merged['STREME_AUROC'].values
_, p = wilcoxon(d_vs_s)
wins_s = (d_vs_s > 0.001).sum()
loss_s = (d_vs_s < -0.001).sum()
print(f'DALE vs STREME: Δ = +{d_vs_s.mean():.4f}, Wilcoxon p = {p:.2e} ({wins_s}W/{loss_s}L)')

d_vs_m = merged['DALE_AUROC'].values - merged['MEME_AUROC'].values
wins_m = (d_vs_m > 0.001).sum()
loss_m = (d_vs_m < -0.001).sum()
_, pm = wilcoxon(d_vs_m) if len(d_vs_m) > 5 else (0, 1)
print(f'DALE vs MEME:   Δ = +{d_vs_m.mean():.4f}, Wilcoxon p = {pm:.2e} ({wins_m}W/{loss_m}L)')

# Figure
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel A: DALE vs STREME scatter
ax = axes[0]
o = merged['DALE_AUROC'].values
s = merged['STREME_AUROC'].values
colors = np.where(o > s, MD_COLOR, ST_COLOR)
ax.scatter(s, o, alpha=0.7, s=40, c=colors, edgecolors='black', linewidths=0.5)
ax.plot([0.4, 1.0], [0.4, 1.0], 'k--', alpha=0.3)
ax.set_xlabel('STREME AUROC', fontsize=11)
ax.set_ylabel('DALE AUROC', fontsize=11)
ax.set_title(f'DALE vs STREME ({wins_s}W/{loss_s}L)', fontsize=12, fontweight='bold')
for _, row in merged.iterrows():
    d = row['DALE_AUROC'] - row['STREME_AUROC']
    if abs(d) > 0.15:
        ax.annotate(row['TF'], (row['STREME_AUROC'], row['DALE_AUROC']), fontsize=8)

# Panel B: DALE vs MEME scatter
ax = axes[1]
m = merged['MEME_AUROC'].values
colors = np.where(o > m, MD_COLOR, MM_COLOR)
ax.scatter(m, o, alpha=0.7, s=40, c=colors, edgecolors='black', linewidths=0.5)
ax.plot([0.4, 1.0], [0.4, 1.0], 'k--', alpha=0.3)
ax.set_xlabel('MEME AUROC', fontsize=11)
ax.set_ylabel('DALE AUROC', fontsize=11)
ax.set_title(f'DALE vs MEME ({wins_m}W/{loss_m}L)', fontsize=12, fontweight='bold')
for _, row in merged.iterrows():
    d = row['DALE_AUROC'] - row['MEME_AUROC']
    if abs(d) > 0.15:
        ax.annotate(row['TF'], (row['MEME_AUROC'], row['DALE_AUROC']), fontsize=8)

# Panel C: Speed comparison
ax = axes[2]
tools = ['DALE', 'STREME', 'MEME']
times = [merged['DALE_Time'].mean(), merged['STREME_Time'].mean(), merged['MEME_Time'].mean()]
colors = [MD_COLOR, ST_COLOR, MM_COLOR]
bars = ax.bar(tools, times, color=colors, edgecolor='black', width=0.5)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(times)*0.02,
            f'{t:.1f}s', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Time per TF (seconds)', fontsize=11)
ax.set_title('Speed Comparison', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.set_ylim(bottom=0.1)

plt.tight_layout()
plt.savefig('head_to_head_results.png', dpi=150, bbox_inches='tight')
plt.show()



## 5. Per-TF Details


In [ ]:
display_cols = ['TF', 'DALE_AUROC', 'STREME_AUROC', 'MEME_AUROC',
                       'DALE_Time', 'STREME_Time', 'MEME_Time']
detail = merged[display_cols].copy()
detail['DALE_lead'] = (detail['DALE_AUROC'] > detail['STREME_AUROC']) & (detail['DALE_AUROC'] > detail['MEME_AUROC'])
detail = detail.sort_values('DALE_AUROC', ascending=False)
print(detail.to_string(index=False))
print(f'\nDALE leads on {detail["DALE_lead"].sum()} of {len(detail)} TFs')



---

**Reproducibility checklist:**
- ✅ All three tools run from source with default parameters
- ✅ Same input sequences (12 ENCODE K562 ChIP-seq TFs)
- ✅ Same AUROC metric on dinucleotide-shuffled negatives (seed=42)
- ✅ No pre-computed results — everything runs live
- ✅ DALE binary is static, zero dependencies; STREME/MEME via conda
- ✅ Full paper + code: [github.com/Travis42/little-scientist-dale](https://github.com/Travis42/little-scientist-dale)
